---

# Fishora: Gated ViT-B Pipeline

*Two rejection gates on the frozen ViT-B embedding, so the species classifier only answers for fish it can know.*

---

## Fishora Team

- Member 1
- Member 2
- Member 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Species Exports**](#3)
4. [**Embeddings**](#4)
5. [**Gate 1: Fish Gate**](#5)
6. [**Gate 2: Known-Species Gate**](#6)
7. [**Gated Exports**](#7)
8. [**Validation Results**](#8)
9. [**Export**](#9)


---

# Introduction <a name="1"></a>

---

## Overview

*Fishora's species classifier is a closed-set model: it always answers with one of eleven species. The evaluation suite showed what that costs. With the production threshold of 0.0, 100% of non-fish photos and of fish from other species are accepted, and for ViT-B 37.6% of unknown species are still accepted at the operating point that keeps 95% of real fish.*

*A human confirms every species, but a confident wrong label is exactly what a busy operator confirms without looking. The pipeline therefore needs to refuse inputs it cannot know before it offers a species.*

## Aim

*Add two gates in front of the ViT-B species head, both computed from the same backbone pass: Gate 1 decides whether the photo shows a fish at all, and Gate 2 decides whether the fish resembles any fish the model was trained on. A rejected photo goes to manual species entry.*

*The gates are built three times, once per species export: for the two cross-fitted models (held-out folds A and B), which the evaluation suite scores on the locked test sets, and for the final model trained on all field photos, which is deployed. Every threshold is chosen on validation data in this notebook; no locked test photo is loaded here.*

## Metric

***True positive rate (TPR)** is the share of real fish that a gate lets through. It sets the operating point, because rejecting a real fish costs the operator a manual entry.*

$$\text{TPR} = \frac{\text{fish accepted}}{\text{all fish}}$$

***Rejection rate** is the share of out-of-scope inputs a gate stops, reported per input type on validation groups (non-fish queries and fish taxa) that were never used for training.*

***AUROC** measures how well a score separates in-scope fish from out-of-scope inputs over all thresholds. It is used to choose `k` for Gate 2.*

## Dataset

*Data sourced from Kaggle, specifically the Fishora dataset at: https://www.kaggle.com/datasets/athillazaidan/fishora, and from the private `fishora_eval_kit` dataset packed by `evaluation/cv/kaggle/pack_kit.py`.*

*Three sources are used. The Fishora train and validation splits (3,932 and 847 images) are the in-scope fish of the dataset. The 258 screened field photos, split into locked folds by photographer, are in-scope fish from real conditions. The gate training pool, fetched by `evaluation/cv/prep/fetch_gate_data.py`, holds fish of 20 taxa outside both the 11 classes and the unknown-species test set, and non-fish photos for queries disjoint from the test non-fish queries, all leak-checked against the test photos.*

*The dataset test split and the kit's `ood/` test photos are not used in this notebook.*

```
fishora_eval_kit/evaluation/cv/
├── protocol/
|    ├── field_folds.csv       fold A / B per field photo, locked
|    └── field_folds.lock
└── data/
     ├── field/<species>/      258 in-scope field photos
     ├── gate_train/
     |    ├── fish/<taxon>/    fish outside every test set
     |    └── nonfish/         non-fish, queries outside the test set
     └── gate_attribution.csv
```

## Approach: Two Gates on One Embedding

*The backbone is frozen, so every photo is embedded once and all three components read the same 768-dimensional CLS embedding, L2-normalised for the gates. Serving cost stays one backbone pass (about 267 ms on CPU) plus a dot product.*

```
Photo -> ViT-B/16 DINOv3 (frozen) -> CLS embedding [768]
    |
    +-- Gate 1: logistic head -> P(fish)          reject if < fish_threshold
    +-- Gate 2: k-th nearest training embedding   reject if distance > knn_threshold
    +-- Species head (unchanged) -> 11 classes
```

*Gate 1 is trained with fish as positives (dataset, field photos of the allowed folds, fish of other taxa) and non-fish as negatives. Fish of other taxa are positives on purpose: Gate 1 answers only whether a fish is present, and leaves unknown species to Gate 2.*

*Gate 2 has no trained weights. It stores the embeddings of the dataset train split and the allowed field photos, and scores a photo by its cosine distance to the k-th nearest one (the deep nearest-neighbour detector of Sun et al., 2022). Its threshold is set on field photos scored leave-one-photographer-out, so a photo never finds itself or its photographer's other photos in the bank.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

Enable a GPU (T4 or P100) in the session settings and add three inputs: the Fishora dataset, the `fishora_eval_kit` dataset (rebuilt with the gate pool), and the species exports: the cross-fitted ViT-B folds from the field-probe notebook and the final ViT-B from the final-model notebook. Embedding extraction takes about five minutes on a T4; everything after it runs on cached embeddings in seconds.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q -U "timm>=1.0.20"

## Import Libraries

In [ ]:
import hashlib
import json
import os
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

print('timm', timm.__version__, '| torch', torch.__version__)

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

## Settings

`FIELD_VAL_SHARE` and `SEED` must match the species notebooks, so the field photos a gate validates on are the ones the species head also held out for validation. `FISH_TARGET_TPR` and `KNN_TARGET_TPR` are the operating points: the share of real fish each gate must let through on validation.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    INPUT      = Path('/kaggle/input') if _ON_KAGGLE else Path('../../..').resolve()
    OUTPUT_DIR = Path('/kaggle/working/fishora_gated') if _ON_KAGGLE else Path('../kaggle_outputs/gated/fishora_gated')

    VARIANTS          = ['A', 'B', 'final']
    FIELD_VAL_SHARE   = 0.2
    GATE_VAL_SHARE    = 0.2
    BATCH_SIZE        = 64
    NUM_WORKERS       = 4

    FISH_C            = 1.0
    FISH_TARGET_TPR   = 0.99
    K_GRID            = [1, 3, 5, 10, 20]
    KNN_TARGET_TPR    = 0.95

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| output:', CFG.OUTPUT_DIR)

## Load Dataset

The dataset root is the directory holding `cleaned/` and `splits/`. The kit is the directory holding `evaluation/cv/protocol/field_folds.csv` together with the gate pool, and the fold file is checked against its lock before use. Gate photos marked `drop` in the visual screening (`gate_screening.csv`: non-fish photos that show a fish, fish photos that are fragments or tiny) are removed. Only the train and validation splits are read.

In [ ]:
def find_dirs(pred):
    hits = [p for p in CFG.INPUT.rglob('*') if p.is_dir() and 'kaggle_outputs' not in p.parts and pred(p)]
    return sorted(hits, key=lambda p: len(str(p)))

DATA_ROOT = find_dirs(lambda p: (p / 'cleaned').is_dir() and (p / 'splits').is_dir())[0]
KIT = find_dirs(lambda p: (p / 'evaluation/cv/protocol/field_folds.csv').exists()
                and (p / 'evaluation/cv/data/gate_attribution.csv').exists())[0]
KIT_DATA = KIT / 'evaluation' / 'cv' / 'data'

def load_split(name):
    df = pd.read_csv(DATA_ROOT / 'splits' / f'{name}.csv')
    return pd.DataFrame({'path': [str(DATA_ROOT / p) for p in df['clean_path']],
                         'label': df['normalized_label'], 'source': f'dataset_{name}', 'group': name})

folds_path = KIT / 'evaluation/cv/protocol/field_folds.csv'
lock = dict(l.split('=', 1) for l in (KIT / 'evaluation/cv/protocol/field_folds.lock').read_text().splitlines() if '=' in l)
assert hashlib.sha256(folds_path.read_bytes()).hexdigest() == lock['field_folds.csv sha256'], 'folds changed after lock'
folds = pd.read_csv(folds_path)
field_df = pd.DataFrame({'path': [str(KIT_DATA / f) for f in folds['file']], 'label': folds['label'],
                         'source': 'field', 'group': folds['observer'], 'fold': folds['fold']})

gate = pd.read_csv(KIT_DATA / 'gate_attribution.csv')
screen = pd.read_csv(KIT_DATA / 'gate_screening.csv')
gate = gate[gate['file'].isin(screen.loc[screen['decision'] == 'keep', 'file'])]
gate_df = pd.DataFrame({'path': [str(KIT_DATA / f) for f in gate['file']], 'label': gate['label'],
                        'source': gate['set'], 'group': gate['label']})

items = pd.concat([load_split('train'), load_split('val'), field_df, gate_df], ignore_index=True)
missing = [p for p in items['path'] if not Path(p).exists()]
assert not missing, f'{len(missing)} images missing, first: {missing[0]}'
print('dataset:', DATA_ROOT, '| kit:', KIT)
items.groupby('source').size().rename('images').to_frame()

---

# Species Exports <a name="3"></a>

---

The gates do not change the species heads. Each variant starts from an existing ViT-B export: the cross-fitted fold models identify themselves with `heldout_fold` in their config, and the final model with `field_training: all`. Because the backbone was frozen in every species run, all three exports must hold the identical backbone, which is checked here so that one embedding pass serves all three variants.

In [ ]:
def discover_exports():
    found = {}
    for cfg_path in CFG.INPUT.rglob('inference_config.json'):
        if 'kaggle_outputs' in cfg_path.parts and CFG._ON_KAGGLE:
            continue
        cfg = json.loads(cfg_path.read_text())
        if not cfg.get('model_name', '').startswith('vit_base_patch16_dinov3') or cfg.get('gate'):
            continue
        if cfg.get('field_training') == 'all':
            found.setdefault('final', cfg_path.parent)
        elif cfg.get('heldout_fold') in ('A', 'B'):
            found.setdefault(cfg['heldout_fold'], cfg_path.parent)
    missing = [v for v in CFG.VARIANTS if v not in found]
    assert not missing, f'species exports missing for {missing}'
    return {v: found[v] for v in CFG.VARIANTS}

EXPORTS = discover_exports()
states = {v: torch.load(d / 'model_state_dict.pt', map_location='cpu', weights_only=True) for v, d in EXPORTS.items()}
backbone_keys = [k for k in states['A'] if not k.startswith('head.')]
for v in CFG.VARIANTS[1:]:
    same = all(torch.equal(states['A'][k], states[v][k]) for k in backbone_keys)
    assert same, f'backbone of {v} differs from A'
del states
for v, d in EXPORTS.items():
    print(f'{v:6s} {d}')
print(f'backbone identical across variants ({len(backbone_keys)} tensors)')

---

# Embeddings <a name="4"></a>

---

Every image is embedded once through the gated wrapper itself, loaded from the first export, so the notebook uses the exact preprocessing and pooling that the service will run. Extraction runs in fp32 to keep the thresholds valid for the fp32 CPU service.

## Gated Wrapper

`GATED_PY` is `evaluation/cv/kaggle/gated_inference.py` from the repository, embedded verbatim. It is written into every export as `inference.py`, and `score_batch` returns the backbone embedding path used below.

In [ ]:
GATED_PY = r'''import json
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image, ImageOps
from torchvision import transforms


class ResizePad:
    def __init__(self, size, fill):
        self.size = int(size)
        self.fill = tuple(fill)

    def __call__(self, image):
        image = ImageOps.exif_transpose(image).convert("RGB")
        w, h = image.size
        scale = min(self.size / w, self.size / h)
        nw = max(1, int(round(w * scale)))
        nh = max(1, int(round(h * scale)))
        image = image.resize((nw, nh), Image.Resampling.BICUBIC)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(image, ((self.size - nw) // 2, (self.size - nh) // 2))
        return canvas


class FishoraClassifier:
    """Species classifier behind two rejection gates, sharing one backbone pass.

    Gate 1 (fish gate): logistic head on the L2-normalised embedding, rejects
    photos that do not show a fish. Gate 2 (known-species gate): cosine distance
    to the k-th nearest training embedding, rejects fish unlike any training
    photo. Without gate files in the export this behaves like the plain wrapper.
    """

    def __init__(self, export_dir, device=None):
        export_dir = Path(export_dir)
        with open(export_dir / "inference_config.json", "r", encoding="utf-8") as f:
            self.cfg = json.load(f)

        self.device = torch.device(device if device else ("cuda" if torch.cuda.is_available() else "cpu"))

        self.model = timm.create_model(self.cfg["model_name"], pretrained=False, num_classes=self.cfg["num_classes"])
        try:
            self.model.reset_classifier(num_classes=self.cfg["num_classes"], global_pool="token")
        except Exception:
            pass
        state = torch.load(export_dir / "model_state_dict.pt", map_location="cpu", weights_only=True)
        self.model.load_state_dict(state)
        self.model = self.model.to(self.device).eval()

        self.transform = transforms.Compose([
            ResizePad(self.cfg["img_size"], self.cfg["fill_rgb"]),
            transforms.ToTensor(),
            transforms.Normalize(self.cfg["mean"], self.cfg["std"]),
        ])

        self.gate = self.cfg.get("gate")
        if self.gate:
            head = torch.load(export_dir / self.gate["fish_head_file"], map_location="cpu", weights_only=True)
            self.fish_w = head["weight"].float().to(self.device)
            self.fish_b = head["bias"].float().to(self.device)
            bank = np.load(export_dir / self.gate["knn_bank_file"]).astype(np.float32)
            self.bank = torch.from_numpy(bank).to(self.device)

    @torch.inference_mode()
    def score_batch(self, x):
        """Logits plus gate scores for a preprocessed batch. The suite calls this
        directly so evaluation and serving share one code path."""
        feats = self.model.forward_head(self.model.forward_features(x), pre_logits=True).float()
        logits = self.model.get_classifier()(feats).float()
        if not self.gate:
            return logits, None, None
        z = torch.nn.functional.normalize(feats, dim=1)
        fish_prob = torch.sigmoid(z @ self.fish_w.T + self.fish_b).squeeze(1)
        k = int(self.gate["knn_k"])
        sims = z @ self.bank.T
        knn_dist = 1.0 - sims.topk(k, dim=1).values[:, -1]
        return logits, fish_prob, knn_dist

    @torch.inference_mode()
    def predict(self, image, top_k=3):
        if isinstance(image, (str, Path)):
            image = Image.open(image)

        x = self.transform(image.convert("RGB")).unsqueeze(0).to(self.device)
        logits, fish_prob, knn_dist = self.score_batch(x)
        probs = torch.softmax(logits / float(self.cfg["temperature"]), dim=1)[0]

        k = min(top_k, len(self.cfg["classes"]))
        values, indices = probs.topk(k)
        candidates = [
            {"label": self.cfg["classes"][int(idx)], "confidence": float(value)}
            for value, idx in zip(values.cpu(), indices.cpu())
        ]
        best = candidates[0]
        threshold = float(self.cfg["abstain_threshold"])
        status = "confident_prediction" if best["confidence"] >= threshold else "low_confidence_human_verification_required"

        result = {"status": status, "prediction": best, "top_candidates": candidates, "threshold": threshold}
        if self.gate:
            fish_prob, knn_dist = float(fish_prob[0]), float(knn_dist[0])
            if fish_prob < self.gate["fish_threshold"]:
                result["status"] = "rejected_not_fish"
            elif knn_dist > self.gate["knn_threshold"]:
                result["status"] = "rejected_unknown_species"
            result["gate"] = {
                "fish_prob": fish_prob, "fish_threshold": self.gate["fish_threshold"],
                "knn_distance": knn_dist, "knn_threshold": self.gate["knn_threshold"],
            }
        return result
'''

def load_wrapper(export_dir):
    ns = {}
    exec(compile(GATED_PY, str(export_dir / 'inference.py'), 'exec'), ns)
    return ns['FishoraClassifier'](export_dir, device=str(DEVICE))

clf = load_wrapper(EXPORTS['A'])
print('embedding dim:', clf.model.num_features)

## Extraction

Embeddings are cached to `embeddings.npy` next to `items.csv`, so a re-run after a disconnect skips extraction. Unreadable images would raise here rather than being skipped silently.

In [ ]:
class ImageList(Dataset):
    def __init__(self, paths, tf):
        self.paths, self.tf = list(paths), tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as img:
            return self.tf(img.convert('RGB'))

@torch.inference_mode()
def embed(paths):
    loader = DataLoader(ImageList(paths, clf.transform), batch_size=CFG.BATCH_SIZE, num_workers=CFG.NUM_WORKERS)
    out = []
    for x in tqdm(loader, desc='embed'):
        x = x.to(DEVICE)
        feats = clf.model.forward_head(clf.model.forward_features(x), pre_logits=True).float()
        out.append(torch.nn.functional.normalize(feats, dim=1).cpu().numpy())
    return np.concatenate(out)

emb_path = CFG.OUTPUT_DIR / 'embeddings.npy'
items_path = CFG.OUTPUT_DIR / 'items.csv'
if emb_path.exists() and items_path.exists() and pd.read_csv(items_path)['path'].tolist() == items['path'].tolist():
    Z = np.load(emb_path)
    print('loaded cached embeddings')
else:
    Z = embed(items['path'])
    np.save(emb_path, Z)
    items.to_csv(items_path, index=False)
print('embeddings:', Z.shape)

## Validation Groups

Validation is split by group, never by image. Field photos use the same photographer split as the species notebooks (same seed, same share), and the gate pool holds out whole taxa and whole non-fish queries, so the validation numbers show how the gates handle kinds of input they were not trained on.

In [ ]:
def split_groups(groups, share, seed=CFG.SEED):
    groups = sorted(set(groups))
    random.Random(seed).shuffle(groups)
    return set(groups[: int(round(len(groups) * share))])

def field_pool(variant):
    return items['source'].eq('field') & (items['fold'].ne(variant) if variant in ('A', 'B') else True)

def field_val_mask(variant):
    pool = field_pool(variant)
    val_obs = split_groups(items.loc[pool, 'group'], CFG.FIELD_VAL_SHARE)
    return pool & items['group'].isin(val_obs)

gate_val_groups = {s: split_groups(items.loc[items['source'].eq(s), 'group'], CFG.GATE_VAL_SHARE)
                   for s in ('gate_fish', 'gate_nonfish')}
is_gate_val = pd.Series(False, index=items.index)
for s, g in gate_val_groups.items():
    is_gate_val |= items['source'].eq(s) & items['group'].isin(g)
for s, g in gate_val_groups.items():
    print(f'{s} validation groups: {sorted(g)}')

---

# Gate 1: Fish Gate <a name="5"></a>

---

Gate 1 is a logistic regression on the normalised embedding with balanced class weights, since fish outnumber non-fish about eight to one. For each variant, the field photos of the held-out fold are excluded entirely, so the fold models never see the photos they are later scored on.

## Training

Positives are the dataset train split, the training field photos of the variant, and the training taxa of the other-fish pool. Negatives are the training non-fish queries. The validation set mirrors this with the dataset validation split, the validation photographers, and the held-out taxa and queries.

In [ ]:
def gate1_sets(variant):
    fval = field_val_mask(variant)
    ftrain = field_pool(variant) & ~fval
    src = items['source']
    train = src.eq('dataset_train') | ftrain | (src.isin(['gate_fish', 'gate_nonfish']) & ~is_gate_val)
    val = src.eq('dataset_val') | fval | (src.isin(['gate_fish', 'gate_nonfish']) & is_gate_val)
    return train, val

def fit_gate1(variant):
    train, _ = gate1_sets(variant)
    y = (items.loc[train, 'source'] != 'gate_nonfish').astype(int).to_numpy()
    model = LogisticRegression(C=CFG.FISH_C, class_weight='balanced', max_iter=5000)
    model.fit(Z[train.to_numpy()], y)
    return model

gate1 = {v: fit_gate1(v) for v in CFG.VARIANTS}
for v, m in gate1.items():
    train, val = gate1_sets(v)
    print(f'{v:6s} train {int(train.sum())} / val {int(val.sum())} images')

## Threshold

The threshold is the lowest value that still lets `FISH_TARGET_TPR` of the fish through in every validation source separately (dataset, field, other taxa). Taking the minimum over sources keeps the easy dataset photos from setting a threshold that rejects real field photos.

In [ ]:
def fish_prob(model, mask):
    return model.predict_proba(Z[mask.to_numpy()])[:, 1]

def gate1_threshold(variant):
    _, val = gate1_sets(variant)
    per_source = {}
    for s in ('dataset_val', 'field', 'gate_fish'):
        p = fish_prob(gate1[variant], val & items['source'].eq(s))
        per_source[s] = float(np.quantile(p, 1 - CFG.FISH_TARGET_TPR))
    return min(per_source.values()), per_source

fish_thr = {}
rows = []
for v in CFG.VARIANTS:
    thr, per_source = gate1_threshold(v)
    fish_thr[v] = thr
    _, val = gate1_sets(v)
    row = {'variant': v, 'fish_threshold': thr}
    for s in ('dataset_val', 'field', 'gate_fish', 'gate_nonfish'):
        p = fish_prob(gate1[v], val & items['source'].eq(s))
        row[f'{s}_pass'] = float((p >= thr).mean())
    rows.append(row)
gate1_val = pd.DataFrame(rows).set_index('variant')
gate1_val.style.format('{:.3f}')

#### Insights

> Fill in after the run: the non-fish pass rate on held-out queries (`gate_nonfish_pass`, lower is better) against the field pass rate (`field_pass`, which should stay near `FISH_TARGET_TPR`).

> Other-fish taxa should pass Gate 1 at a high rate. If they do not, Gate 1 has learned "one of our species" instead of "a fish", and would overlap with Gate 2.

---

# Gate 2: Known-Species Gate <a name="6"></a>

---

Gate 2 stores a bank of training embeddings and scores a photo by its cosine distance to the k-th nearest bank entry. A fish of a species the model knows lands close to many training photos of that species; a fish of another species does not, even when the species head confidently picks its nearest class.

## Bank and Leave-One-Photographer-Out Scores

The bank holds the dataset train split and every field photo the variant may see. Scoring those same field photos against the bank would be circular, so each field photo is scored with every bank entry from its own photographer masked out. Other-fish photos are never in the bank and act as unknown species.

In [ ]:
def bank_mask(variant):
    return items['source'].eq('dataset_train') | field_pool(variant)

def kth_distance(query, bank, k, exclude=None):
    sims = torch.from_numpy(query) @ torch.from_numpy(bank).T
    if exclude is not None:
        sims[torch.from_numpy(exclude)] = -2.0
    return (1.0 - sims.topk(k, dim=1).values[:, -1]).numpy()

def gate2_scores(variant, k):
    bm = bank_mask(variant).to_numpy()
    bank = Z[bm].astype(np.float16).astype(np.float32)
    bank_group = items.loc[bm, 'group'].to_numpy()
    bank_is_field = items.loc[bm, 'source'].eq('field').to_numpy()
    fp = field_pool(variant).to_numpy()
    field_groups = items.loc[fp, 'group'].to_numpy()
    exclude = (field_groups[:, None] == bank_group[None, :]) & bank_is_field[None, :]
    scores = {'field_loo': kth_distance(Z[fp], bank, k, exclude)}
    for s in ('dataset_val', 'gate_fish', 'gate_nonfish'):
        scores[s] = kth_distance(Z[items['source'].eq(s).to_numpy()], bank, k)
    return scores, bank

def auroc(in_scores, out_scores):
    y = np.r_[np.zeros(len(in_scores)), np.ones(len(out_scores))]
    return float(roc_auc_score(y, np.r_[in_scores, out_scores]))

## Choosing k

`k` is chosen per variant by AUROC between the field photos (leave-one-photographer-out) and the other-fish taxa, which is Gate 2's actual task. Only validation data enters this choice.

In [ ]:
k_rows = []
for v in CFG.VARIANTS:
    for k in CFG.K_GRID:
        s, _ = gate2_scores(v, k)
        k_rows.append({'variant': v, 'k': k, 'auroc_other_fish': auroc(s['field_loo'], s['gate_fish']),
                       'auroc_nonfish': auroc(s['field_loo'], s['gate_nonfish'])})
k_table = pd.DataFrame(k_rows)
best_k = k_table.loc[k_table.groupby('variant')['auroc_other_fish'].idxmax()].set_index('variant')['k'].to_dict()
print('chosen k:', best_k)
k_table.pivot(index='k', columns='variant', values='auroc_other_fish').style.format('{:.3f}')

## Threshold

The distance threshold is the `KNN_TARGET_TPR` quantile of the leave-one-photographer-out field distances, so that share of real field fish passes Gate 2 by construction. The dataset validation split is reported as a sanity check and should pass almost entirely.

In [ ]:
knn_thr, banks, rows = {}, {}, []
for v in CFG.VARIANTS:
    s, banks[v] = gate2_scores(v, best_k[v])
    knn_thr[v] = float(np.quantile(s['field_loo'], CFG.KNN_TARGET_TPR))
    rows.append({'variant': v, 'k': best_k[v], 'knn_threshold': knn_thr[v],
                 **{f'{name}_pass': float((d <= knn_thr[v]).mean()) for name, d in s.items()}})
gate2_val = pd.DataFrame(rows).set_index('variant')
gate2_val.style.format('{:.3f}')

#### Insights

> Fill in after the run: the other-fish pass rate (`gate_fish_pass`, lower is better) at the fixed field pass rate, compared with the 37.6% unknown-species acceptance of max-softmax at the same 95% operating point in the evaluation suite.

> These are validation numbers on taxa chosen for the gate pool. The locked unknown-species test set (lele, kakap, bawal, cakalang, kerapu, patin, gurame) is scored only by the evaluation suite.

---

# Gated Exports <a name="7"></a>

---

Each gated export is the species export plus three changes: `inference.py` becomes the gated wrapper, `fish_head.pt` holds the Gate 1 weights, and `knn_bank.npy` holds the Gate 2 bank in float16. The config keeps `heldout_fold` or `field_training` from the species export, so the evaluation suite still scores each fold model only on its held-out field photos.

## Writing the Exports

The config records the thresholds, `k`, the operating points they were set for, and the validation pass rates, so every number in the export can be traced back to this notebook.

In [ ]:
RUN_NAME = {'A': 'vit_b_gated_xfA', 'B': 'vit_b_gated_xfB', 'final': 'vit_b_gated_final'}

def write_gated_export(v):
    src, dst = EXPORTS[v], CFG.OUTPUT_DIR / RUN_NAME[v] / 'export'
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(src / 'model_state_dict.pt', dst / 'model_state_dict.pt')
    (dst / 'inference.py').write_text(GATED_PY, encoding='utf-8')
    m = gate1[v]
    torch.save({'weight': torch.tensor(m.coef_, dtype=torch.float32),
                'bias': torch.tensor(m.intercept_, dtype=torch.float32)}, dst / 'fish_head.pt')
    np.save(dst / 'knn_bank.npy', banks[v].astype(np.float16))
    cfg = json.loads((src / 'inference_config.json').read_text())
    cfg['recipe'] = cfg.get('recipe', '') + '+gates_v1'
    cfg['gate'] = {
        'fish_head_file': 'fish_head.pt', 'fish_threshold': fish_thr[v], 'fish_target_tpr': CFG.FISH_TARGET_TPR,
        'knn_bank_file': 'knn_bank.npy', 'knn_k': int(best_k[v]), 'knn_threshold': knn_thr[v],
        'knn_target_tpr': CFG.KNN_TARGET_TPR, 'bank_size': int(len(banks[v])),
        'embedding': 'L2-normalised CLS pre-logits of the frozen backbone',
    }
    cfg['gate_validation'] = {'gate1': gate1_val.loc[v].to_dict(), 'gate2': gate2_val.loc[v].to_dict()}
    (dst / 'inference_config.json').write_text(json.dumps(cfg, indent=2), encoding='utf-8')
    return dst

GATED = {v: write_gated_export(v) for v in CFG.VARIANTS}
for v, d in GATED.items():
    size = sum(f.stat().st_size for f in d.iterdir()) / 2**20
    print(f'{v:6s} {d}  ({size:.0f} MB)')

## Parity Check

Each export is reloaded through the wrapper exactly as the CV service loads it, and its gate scores on sample images are compared with the scores computed in this notebook. The sample covers every source, and the returned statuses show the gates acting on single photos.

In [ ]:
sample = items.groupby('source', group_keys=False).apply(lambda d: d.sample(3, random_state=CFG.SEED))
parity_rows = []
for v, d in GATED.items():
    served = load_wrapper(d)
    for idx, row in sample.iterrows():
        out = served.predict(row['path'])
        z = Z[[idx]]
        expected_fish = float(gate1[v].predict_proba(z)[0, 1])
        expected_knn = float(kth_distance(z, banks[v], best_k[v])[0])
        parity_rows.append({'variant': v, 'source': row['source'], 'status': out['status'],
                            'fish_diff': abs(out['gate']['fish_prob'] - expected_fish),
                            'knn_diff': abs(out['gate']['knn_distance'] - expected_knn)})
    del served
parity = pd.DataFrame(parity_rows)
worst = parity[['fish_diff', 'knn_diff']].max()
print(f"max |fish_prob diff| = {worst['fish_diff']:.2e}, max |knn diff| = {worst['knn_diff']:.2e}")
assert worst.max() < 1e-3, 'export does not reproduce notebook gate scores'
parity.pivot_table(index='source', columns='variant', values='status', aggfunc=lambda s: ', '.join(s))

---

# Validation Results <a name="8"></a>

---

This section combines both gates on validation data only. A photo passes the pipeline when it passes Gate 1 and Gate 2. The locked test results (field, unknown species, non-fish, synthetic) come from the evaluation suite, which scores the fold exports; the numbers here only show that the operating points behave as intended.

## Combined Pass Rates

Field photos are scored leave-one-photographer-out for Gate 2 and with the variant's Gate 1, restricted to validation photographers, which is the closest this notebook can get to unseen field conditions without touching the held-out fold.

In [ ]:
def pipeline_pass(v, mask, knn_scores):
    p1 = fish_prob(gate1[v], mask) >= fish_thr[v]
    p2 = knn_scores <= knn_thr[v]
    return p1, p2

rows = []
for v in CFG.VARIANTS:
    s, _ = gate2_scores(v, best_k[v])
    fp = field_pool(v)
    fval_in_pool = field_val_mask(v)[fp].to_numpy()
    cases = {
        'field (val photographers)': (field_val_mask(v), s['field_loo'][fval_in_pool]),
        'dataset val': (items['source'].eq('dataset_val'), s['dataset_val']),
        'other fish (held-out taxa)': (items['source'].eq('gate_fish') & is_gate_val,
                                       s['gate_fish'][is_gate_val[items['source'].eq('gate_fish')].to_numpy()]),
        'non-fish (held-out queries)': (items['source'].eq('gate_nonfish') & is_gate_val,
                                        s['gate_nonfish'][is_gate_val[items['source'].eq('gate_nonfish')].to_numpy()]),
    }
    for name, (mask, knn) in cases.items():
        p1, p2 = pipeline_pass(v, mask, knn)
        rows.append({'variant': v, 'input': name, 'n': int(mask.sum()), 'gate1_pass': p1.mean(),
                     'gate2_pass': p2.mean(), 'pipeline_pass': (p1 & p2).mean()})
combined = pd.DataFrame(rows)
combined.to_csv(CFG.OUTPUT_DIR / 'gate_validation.csv', index=False)
combined.pivot(index='input', columns='variant', values='pipeline_pass').style.format('{:.1%}')

## Score Distributions

The histograms show where each input type falls relative to the thresholds for the final variant. Overlap near a threshold is where the gates will make mistakes.

In [ ]:
v = 'final'
s, _ = gate2_scores(v, best_k[v])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, src in [('dataset val', 'dataset_val'), ('field', 'field'), ('other fish', 'gate_fish'), ('non-fish', 'gate_nonfish')]:
    mask = field_pool(v) if src == 'field' else items['source'].eq(src)
    axes[0].hist(fish_prob(gate1[v], mask), bins=40, range=(0, 1), alpha=0.55, density=True, label=name)
axes[0].axvline(fish_thr[v], color='k', ls='--', lw=1)
axes[0].set(title='Gate 1: P(fish), final variant', xlabel='P(fish)', ylabel='density')
for name, key in [('dataset val', 'dataset_val'), ('field (LOO)', 'field_loo'), ('other fish', 'gate_fish'), ('non-fish', 'gate_nonfish')]:
    axes[1].hist(s[key], bins=40, alpha=0.55, density=True, label=name)
axes[1].axvline(knn_thr[v], color='k', ls='--', lw=1)
axes[1].set(title=f'Gate 2: k-th NN distance (k={best_k[v]}), final variant', xlabel='cosine distance')
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(CFG.OUTPUT_DIR / 'gate_distributions.png', dpi=150)
plt.show()

#### Insights

> Fill in after the run: the share of validation field photos the full pipeline sends to manual entry (the cost), against the share of held-out non-fish and other-fish taxa it stops (the benefit).

> The fold variants and the final variant should agree closely. A large gap would mean the thresholds depend on which photographers happen to be in the bank.

---

# Export <a name="9"></a>

---

Each gated export is zipped separately. The two fold exports go to the evaluation suite notebook as an input; the final export is what `FISHORA_CV_EXPORT_DIR` points at in deployment. Embeddings are not archived because they can be rebuilt in minutes.

In [ ]:
archives = []
for v, run in RUN_NAME.items():
    archives.append(shutil.make_archive(str(CFG.OUTPUT_DIR.parent / f'fishora_{run}'), 'zip',
                                        root_dir=CFG.OUTPUT_DIR, base_dir=run))
for f in ('gate_validation.csv', 'gate_distributions.png'):
    shutil.copy(CFG.OUTPUT_DIR / f, CFG.OUTPUT_DIR.parent / f)
for a in archives:
    print(f'{Path(a).name:40s} {Path(a).stat().st_size / 2**20:8.1f} MB')